# Part 2: PCE-based Sobol indices

## 2.0 Connecting to UQ[py]Lab

Execute the following cell to connect to UQpyLab.

In [ ]:
from uqpylab import sessions
import numpy as np
import matplotlib.pyplot as plt

The following cell needs to be executed only once. You find your token in your profile on the UQ[py]Lab website: https://uqpylab.uq-cloud.io/

In [ ]:
# myToken = 'XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX' # REPLACE THIS WITH YOUR TOKEN
# UQCloud_instance = 'https://uqcloud.ethz.ch' # The UQCloud instance to use
# # Start the session
# mySession = sessions.cloud(host=UQCloud_instance, token=myToken, force_restart=True)
# # Get a convenient handle to the command line interface
# uq = mySession.cli
# mySession.save_config() # to save and restore the session later

In the future, you can instead uncomment and execute the following cell to connect to UQ[py]Lab.

In [ ]:
# Start the session
mySession = sessions.cloud()
# Get a convenient handle to the command line interface
uq = mySession.cli
# Reset the session
mySession.reset()

In [ ]:
# Set the random seed for reproducibility
uq.rng(100,'twister');

## 2.1 Define model and input: Flood model

In [ ]:
# Define the model properties
ModelOpts = {
    'Type': 'Model', 
    'ModelFun':'flood.model'
}

# Create the model
myModel = uq.createModel(ModelOpts)

In [ ]:
# Define the marginal properties
InputOpts = {
    'Marginals': [
        {
            'Name': 'Q',
            'Type': 'Gumbel',
            'Parameters': [1013, 558],
            'Bounds': [500, 3000]
        },
        {
            'Name':'Ks',
            'Type': 'Gaussian',
            'Parameters': [30, 8],
            'Bounds': [15, 1e12]
        },
        {
            'Name': 'Zv',
            'Type': 'Triangular',
            'Parameters': [49, 50, 51]
        },
        {
            'Name': 'Zm',
            'Type': 'Triangular',
            'Parameters': [54, 55, 56]
        },
        {
            'Name': 'Hd',
            'Type': 'Uniform',
            'Parameters': [7, 9]
        },
        {
            'Name': 'Cb',
            'Type': 'Triangular',
            'Parameters': [55, 55.5, 56]
        },
        {
            'Name': 'L',
            'Type': 'Triangular',
            'Parameters': [4990, 5000, 5010]
        },
        {
            'Name': 'B',
            'Type': 'Triangular',
            'Parameters': [295, 300, 305]
        }]
}

# Create the input
myInput = uq.createInput(InputOpts)

## 2.2 PCE with UQ[py]Lab
To create a PCE surrogate for the strip foundation model, we define a dictionary:

In [ ]:
MetaOpts= {
    "Type": "Metamodel",
    "MetaType": "PCE",
}

Specify the model and the input:

In [ ]:
MetaOpts["FullModel"] = myModel["Name"] # pass the name of the model (a string)
MetaOpts["Input"] = myInput["Name"]     # pass the name of the input (a string)

### 2.2.1 Ordinary Least Squares
To compute the PCE using Ordinary Least Squares (OLS), specify the method `"OLS"`:

In [ ]:
MetaOpts["Method"] = "OLS"

Furthermore, we need to specify the degree and the experimental design, here 1000 points sampled by LHS:

In [ ]:
MetaOpts["Degree"] = 3
MetaOpts["ExpDesign"] = {
    "NSamples": 1000,
    "Sampling": "LHS"
}

Create the OLS-based PCE metamodel:

In [ ]:
myPCE_OLS = uq.createModel(MetaOpts)

In [ ]:
uq.print(myPCE_OLS)

<div class="alert alert-block alert-success">
    <b>Task:</b> Increase the degree to $p=4$ and $p=5$, and monitor the Leave-one-out (LOO) error. What do you observe? Can you explain why?
</div>

<div class="alert alert-block alert-success">
    <b>Task:</b> Explore the object myPCE_OLS: it is also a dictionary! Try to find the following quantities: 1) the PCE coefficients 2) the LOO
</div>

In [ ]:
print(myPCE_OLS.keys())

In [ ]:
# YOUR CODE HERE:





### 2.2.2 Sparse regression
Instead of OLS, we can use the **sparse regression solver LARS** to compute the coefficients. It includes only the most relevant regressors into the expansion.

In [ ]:
MetaOpts["Method"] = "LARS"

Sparse regression solvers need less experimental design points than ordinary least squares. We reduce the experimental design:

In [ ]:
MetaOpts["ExpDesign"] = {
    "NSamples": 200,
    "Sampling": "LHS"
}

We can also let UQ[py]Lab choose the best degree automatically, based on the leave-one-out (LOO) error. This is called **degree adaptivity**.

In [ ]:
MetaOpts["Degree"] = np.arange(3,10).tolist()

In [ ]:
myPCE_LARS = uq.createModel(MetaOpts)
uq.print(myPCE_LARS)

_<b>Related manual:</b> <a href="https://storage.googleapis.com/uqpylab-doc-html/UserManual_PolynomialChaos.html">UQpyLab User Manual: Polynomial Chaos Expansions</a>_

## 2.3 Computing moments from the PCE coefficients

<div class="alert alert-block alert-success">
    <b>Task:</b> Access the PCE coefficients and compute an estimate for the mean and the variance of the output from them, by summing up the coefficients. The relevant formulas are
    \[ \mathbb{E}[Y] = c_\mathbf{0}, \]
    \[ \mathrm{Var}[Y] = \sum_{\mathbf{\alpha} \neq \mathbf{0}} c_\mathbf{\alpha}^2 \]
</div>

In [ ]:
coeffs = myPCE_LARS["PCE"]["Coefficients"]

mean_PCE = # <<< YOUR CODE HERE
var_PCE = # <<< YOUR CODE HERE

print("Mean estimate:", mean_PCE)
print("Variance estimate:", var_PCE)

## 2.4 PCE-based Sobol' indices (manually)

<div class="alert alert-block alert-success">
<b>Task:</b> Use the coefficients of the PCE below to compute the following quantities:
<ol>
<li>The first-order Sobol' index of \( X_1 = Q \)</li>
<li>The total Sobol' index of \( X_1 = Q \)</li>
</ol>

<ul>
<li>What can you say about the importance of \( Q \)?</li>
<li>What can you say about its interaction with other variables?</li>
</ul>
</div>

In [ ]:
# Extract the necessary quantities
coeffs = np.array(myPCE_LARS["PCE"]["Coefficients"], dtype=float)
indices = np.array(myPCE_LARS["PCE"]["Basis"]["Indices"], dtype=int)
print(indices)
D = myPCE_LARS["PCE"]["Moments"]["Var"] # total variance of the output, computed by UQPyLab

In [ ]:
S1 = 0;
ST = 0;

for c,alpha in zip(coeffs, indices):
    if # Insert condition for total Sobol index
        ST += c**2
        
        if # Insert additional condition for first-order Sobol index
            S1 += c**2

S1 /= D;
ST /= D;

print(S1)
print(ST)


## 2.5 Sobol' indices with UQ[py]Lab
Using the PCE from above, we can also let UQ[py]Lab compute the Sobol indices. Do you get the same results as with your manual computation above?

In [ ]:
SobolOpts = {
    'Type': 'Sensitivity',
    'Method': 'Sobol',
    'Model': myPCE_LARS["Name"]
}



mySobolAnalysisPCE = uq.createAnalysis(SobolOpts)
uq.print(mySobolAnalysisPCE)
uq.display(mySobolAnalysisPCE);

<div class="alert alert-block alert-success">
    <b>Tasks:</b>     
    <ul>
<li>Try out how few samples you can use and still get a reasonably accurate estimate for the Sobol' indices. </li>
<li>For this method, is there a guarantee that first-order indices are smaller than total indices?</li>
    </ul>
</div>

_<b>Related manual:</b> <a href="https://storage.googleapis.com/uqpylab-doc-html/UserManual_Sensitivity.html">UQpyLab User Manual: Sensitivity Analysis</a>_

## Terminate the remote UQCloud session

In [ ]:
mySession.quit()